In [ ]:
# test

In [ ]:
from sklearn.model_selection import train_test_split

# Keep only rows with known outcome labels for a clean stratified split.
split_df = training_dataframe.dropna(subset=["Outcome"]).copy()

train_df, test_df = train_test_split(
    split_df,
    test_size=0.2,
    stratify=split_df["Outcome"],
    random_state=1234,
)

print(f"Train rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")

split_distribution = pd.concat(
    {
        "full": split_df["Outcome"].value_counts(normalize=True).mul(100).round(2),
        "train": train_df["Outcome"].value_counts(normalize=True).mul(100).round(2),
        "test": test_df["Outcome"].value_counts(normalize=True).mul(100).round(2),
    },
    axis=1,
).fillna(0)

print("\nOutcome distribution (%):")
display(split_distribution)

# Optional: feature matrix and target vector for modelling
X_train = train_df.drop(columns=["Outcome"])
y_train = train_df["Outcome"]
X_test = test_df.drop(columns=["Outcome"])
y_test = test_df["Outcome"]

In [ ]:
# Encode outcome labels as binary: Good=1, Poor=0

outcome_mapping = {"Poor": 0, "Good": 1}


split_df = split_df.copy()

train_df = train_df.copy()

test_df = test_df.copy()


split_df["Outcome_binary"] = split_df["Outcome"].map(outcome_mapping)

train_df["Outcome_binary"] = train_df["Outcome"].map(outcome_mapping)

test_df["Outcome_binary"] = test_df["Outcome"].map(outcome_mapping)


if train_df["Outcome_binary"].isna().any() or test_df["Outcome_binary"].isna().any():

    unknown_labels = sorted(set(split_df["Outcome"].dropna()) - set(outcome_mapping))

    raise ValueError(f"Unmapped Outcome labels found: {unknown_labels}")


modeling_train_df, numeric_fill_values = prepare_modeling_features(train_df)

modeling_test_df, _ = prepare_modeling_features(
    test_df, numeric_fill_values=numeric_fill_values
)


y_train_binary = modeling_train_df["Outcome_binary"].astype(int)

y_test_binary = modeling_test_df["Outcome_binary"].astype(int)


print("Outcome mapping:", outcome_mapping)

print("Numeric fill values:", numeric_fill_values)

print("y_train_binary distribution:")

print(y_train_binary.value_counts(normalize=True).mul(100).round(2))

print("\ny_test_binary distribution:")

print(y_test_binary.value_counts(normalize=True).mul(100).round(2))


for column_name in ["Sex", "OHCA", "Shockable Rhythm", "TTM"]:

    print(f"\n{column_name} categories in modeling_train_df:")

    print(modeling_train_df[column_name].value_counts(dropna=False))

In [ ]:
import statsmodels.api as sm

import statsmodels.formula.api as smf

# R equivalent:

# glm(Revenue ~ ., data = train_df, family = binomial(link = ...))

# Here, Revenue corresponds to Outcome_binary (Good=1, Poor=0).

exclude_columns = {
    "Outcome",
    "Outcome_binary",
    "source_file",
    "CPC",
    "Patient",
    "Hospital",
}

predictor_columns = [
    col for col in modeling_train_df.columns if col not in exclude_columns
]


# Q("...") handles column names with spaces/special characters.

formula_terms = [f'Q("{col}")' for col in predictor_columns]

formula = "Outcome_binary ~ " + " + ".join(formula_terms)


logit_model = smf.glm(
    formula=formula,
    data=modeling_train_df,
    family=sm.families.Binomial(link=sm.families.links.Logit()),
    missing="raise",
).fit(maxiter=100, tol=1e-8)


print(f"Model training rows used: {len(modeling_train_df)}")

print("Predictors used:", predictor_columns)

In [ ]:
print("\nLogit summary (first table):")
display(logit_model.summary2().tables[0])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Diagnostics for the fitted logit model
influence_logit = logit_model.get_influence()
cooksd_logit = influence_logit.cooks_distance[0]
pearson_logit = logit_model.resid_pearson

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

axes[0].scatter(
    np.arange(len(cooksd_logit)),
    cooksd_logit,
    s=22,
    c=[(0.2, 0.4, 0.8, 0.2)],
)
axes[0].axhline(4 / len(cooksd_logit), color="red", linewidth=2, linestyle="--")
axes[0].set_title("Logit: Cooks distance")
axes[0].set_ylabel("Value")
axes[0].set_xlabel("Observation index")

log_abs_pearson = np.log(np.abs(pearson_logit) + 1e-6)
axes[1].scatter(
    np.arange(len(log_abs_pearson)),
    log_abs_pearson,
    s=22,
    c=[(0.2, 0.4, 0.8, 0.2)],
)
axes[1].axhline(np.log(3), color="red", linewidth=2, linestyle="--")
axes[1].axhline(-np.log(3), color="red", linewidth=2, linestyle="--")
axes[1].set_title("Logit: log(|Pearson residuals|)")
axes[1].set_ylabel("log(|Pearson residuals|)")
axes[1].set_xlabel("Observation index")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd


def clean_term_name(term_name: str) -> str:
    if term_name.startswith('Q("') and term_name.endswith('")'):
        return term_name[3:-2]
    return term_name


def compute_gvif_table(model_result, model_name: str) -> pd.DataFrame:
    """Compute GVIF and adjusted GVIF (GVIF^(1/(2*Df))) per model term."""
    exog_names = model_result.model.exog_names
    design_info = model_result.model.data.design_info

    # Remove intercept from the coefficient correlation matrix.
    kept_names = [name for name in exog_names if name != "Intercept"]
    cov_params = model_result.cov_params().loc[kept_names, kept_names]

    std = np.sqrt(np.diag(cov_params))
    corr_values = cov_params.to_numpy() / np.outer(std, std)
    corr_matrix = pd.DataFrame(corr_values, index=kept_names, columns=kept_names)

    sign_all, logdet_all = np.linalg.slogdet(corr_matrix.to_numpy())
    if sign_all <= 0:
        raise ValueError(
            f"Correlation matrix is not positive definite for {model_name}."
        )

    rows = []
    for term_name, term_slice in design_info.term_name_slices.items():
        if term_name == "Intercept":
            continue

        term_columns = [
            col for col in exog_names[term_slice] if col in corr_matrix.columns
        ]
        if not term_columns:
            continue

        other_columns = [col for col in corr_matrix.columns if col not in term_columns]

        sign_term, logdet_term = np.linalg.slogdet(
            corr_matrix.loc[term_columns, term_columns].to_numpy()
        )
        if sign_term <= 0:
            gvif = np.nan
        else:
            if other_columns:
                sign_other, logdet_other = np.linalg.slogdet(
                    corr_matrix.loc[other_columns, other_columns].to_numpy()
                )
                gvif = (
                    np.nan
                    if sign_other <= 0
                    else np.exp(logdet_term + logdet_other - logdet_all)
                )
            else:
                gvif = np.exp(logdet_term - logdet_all)

        df_term = len(term_columns)
        gvif_adjusted = np.nan if pd.isna(gvif) else gvif ** (1 / (2 * df_term))

        rows.append(
            {
                "Modelis": model_name,
                "Variable": clean_term_name(term_name),
                "Df": df_term,
                "GVIF": gvif,
                "GVIF^(1/(2*Df))": gvif_adjusted,
            }
        )

    return pd.DataFrame(rows)


gvif_long = pd.concat(
    [compute_gvif_table(logit_model, "Logit")],
    ignore_index=True,
)

gvif_table = (
    gvif_long.pivot(index="Modelis", columns="Variable", values="GVIF^(1/(2*Df))")
    .reset_index()
    .rename_axis(None, axis=1)
)

print("GVIF^(1/(2*Df)) table for all variables:")
display(gvif_table.round(3))

print("\nDetailed GVIF output:")
display(gvif_long.round(3))

In [ ]:
from scipy.stats import binomtest, norm

# Evaluate logit model on test data without dropping rows for predictor missingness.

eval_columns = predictor_columns + ["Outcome_binary"]

logit_eval_df = modeling_test_df[eval_columns].copy()


if logit_eval_df["Outcome_binary"].isna().any():

    raise ValueError("Outcome_binary contains missing values in the evaluation set.")


y_true = logit_eval_df["Outcome_binary"].astype(int)

y_prob = logit_model.predict(logit_eval_df)

y_pred = (y_prob >= 0.5).astype(int)


n = len(y_true)

correct = int((y_pred == y_true).sum())

accuracy = correct / n


# Parametric (normal approximation) 95% CI for accuracy.

z_975 = norm.ppf(0.975)

se_accuracy = (accuracy * (1 - accuracy) / n) ** 0.5

acc_ci_low = max(0.0, accuracy - z_975 * se_accuracy)

acc_ci_high = min(1.0, accuracy + z_975 * se_accuracy)


# Dominant-class baseline on the evaluation sample.

dominant_class = int(y_true.mode().iloc[0])

dominant_accuracy = float((y_true == dominant_class).mean())


# One-sided test: H0 accuracy <= dominant_accuracy vs H1 accuracy > dominant_accuracy.

accuracy_test = binomtest(
    k=correct,
    n=n,
    p=dominant_accuracy,
    alternative="greater",
)

p_value_vs_dominant = accuracy_test.pvalue


# Confusion matrix components for positive class = 1 (Good outcome).

tp = int(((y_pred == 1) & (y_true == 1)).sum())

tn = int(((y_pred == 0) & (y_true == 0)).sum())

fp = int(((y_pred == 1) & (y_true == 0)).sum())

fn = int(((y_pred == 0) & (y_true == 1)).sum())


sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float("nan")

precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")

f1_score = (
    2 * sensitivity * precision / (sensitivity + precision)
    if (sensitivity + precision) > 0
    else float("nan")
)


logit_metrics = pd.DataFrame(
    {
        "Metric": [
            "Overall accuracy",
            "Accuracy 95% CI lower",
            "Accuracy 95% CI upper",
            "P-value (accuracy > dominant-class baseline)",
            "Sensitivity (TP/(TP+FN))",
            "Precision (TP/(TP+FP))",
            "F1 score",
        ],
        "Value": [
            accuracy,
            acc_ci_low,
            acc_ci_high,
            p_value_vs_dominant,
            sensitivity,
            precision,
            f1_score,
        ],
    }
)


print(f"Evaluation rows used: {n}")

print(f"Model-ready test rows available: {len(modeling_test_df)}")

print(
    f"Dominant class in evaluation sample: {dominant_class} (baseline accuracy = {dominant_accuracy:.3f})"
)

print(f"Confusion matrix counts: TP={tp}, TN={tn}, FP={fp}, FN={fn}")

display(logit_metrics)